# Pengujian Skenario: 7 Model x 10 Dataset (Prequential Test-Then-Train + Deteksi Drift)

Protokol sesuai skenario pengujian kelompok:

1. **Chunk 1 = warm-up**: melatih model awal, akurasi belum dihitung. Satu chunk = **500 baris**.
2. **Chunk 2 dst**: *test dulu, baru latih* (interleaved test-then-train).
3. **Deteksi drift**: skor CUSUM-style θ atas error-rate chunk vs jendela referensi; alarm bila θ ≥ 3,0.
4. **Dua strategi adaptasi** per model: `continuous` (tanpa reset) dan `reset` (model dibangun ulang saat alarm).
5. **Metrik**: akurasi rata-rata, prequential kappa, jumlah alarm, RT, PDE, FTDE, latensi per chunk.

**7 model** (satu dataset dicoba bergantian oleh ketujuhnya):

| # | Model | Implementasi |
|---|---|---|
| 1 | DT | Hoeffding Tree (river) |
| 2 | HAT | Hoeffding Adaptive Tree (river) |
| 3 | ARF | Adaptive Random Forest, 10 pohon (river) |
| 4 | SVM | linear SGD hinge loss (scikit-learn) |
| 5 | ANN | MLP 1 hidden layer 32 neuron (scikit-learn) |
| 6 | BELS | implementasi ringkas: ensemble lapisan output di atas pemetaan acak bersama, mini-chunk |
| 7 | DACD | implementasi ringkas: window CUSUM internal + pemilihan sampel + retrain MLP |

**Dataset simulasi** (file hasil generate MOA oleh kelompok, di `Dataset Real Word/datasets_synthetic`, 100.000 baris per dataset):

| Dataset | Fitur | Kelas | Drift |
|---|---|---|---|
| sea | 3 | 2 | gradual (ganti fungsi klasifikasi) |
| rbfblips | 20 | 4 | abrupt (beda centroid) |
| led_abrupt | 24 | 10 | abrupt di 50% (permutasi label) |
| led_gradual | 24 | 10 | gradual di 25/50/75% |
| tree | 30 | 10 | abrupt (beda struktur pohon) |
| hyperplane | 10 | 2 | incremental (mag_change) |

**Dataset real-world**: electricity (45.312), kddcup99 (494.020), covertype (581.012), weather BASEL (3.654).

Perkiraan durasi Run All penuh dengan 14 worker: **1-1,5 jam** (dataset terberat menentukan; KDD/Covertype masing-masing sekitar 20-30 menit). Bila ingin lebih cepat lagi: kecilkan `STRATEGIES` jadi satu, turunkan `ARF_TREES`, atau turunkan `SIM_SAMPLES`.

**Tahan mati listrik**: setiap chunk di-checkpoint otomatis ke `hasil_pengujian/checkpoint/`. Jalankan ulang Run All setelah gangguan, dan proses berlanjut dari chunk terakhir yang belum selesai; run yang sudah selesai dilewati. Hapus folder `hasil_pengujian` bila ingin mulai dari nol.

Cara pakai: **Run All**. Sel pertama otomatis menginstal dependensi bila kernel belum memilikinya.

In [ ]:
import importlib.util
import subprocess
import sys

NEED = {
    "river": "river",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
}
missing = [pip_name for mod, pip_name in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Menginstal paket yang kurang:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    print("Selesai. Lanjut ke sel berikutnya.")
else:
    print("Semua dependensi tersedia di:", sys.executable)

In [ ]:
import json
import os
import time
import warnings
from pathlib import Path

import cloudpickle
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from river import forest, metrics, tree
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)

DATA = Path("dataset")
OUT = Path("hasil_pengujian")
OUT.mkdir(exist_ok=True)
CKPT_DIR = OUT / "checkpoint"
CKPT_DIR.mkdir(exist_ok=True)

THETA_SEVERE = 3.0
RECOVERY_RATIO = 0.95
SEED = 42
ARF_TREES = 10
STRATEGIES = ("continuous", "reset")

# dataset simulasi: file hasil generate MOA oleh kelompok (100.000 baris per dataset)
SIM_DIR = Path("Dataset Real Word") / "datasets_synthetic"

# penyesuaian otomatis kemampuan device: sisakan thread untuk sistem/navigasi
RESERVE_THREADS = 2
TOTAL_THREADS = os.cpu_count() or 2
N_WORKERS = max(1, TOTAL_THREADS - RESERVE_THREADS)
WORKERS_HEAVY = min(N_WORKERS, max(4, N_WORKERS - 4))
LOG_DIR = OUT / "logs"
LOG_DIR.mkdir(exist_ok=True)

print(f"CPU terdeteksi: {TOTAL_THREADS} thread | worker: {N_WORKERS} ({RESERVE_THREADS} disisakan)")
print(f"Dataset berat (kddcup99, covertype) pakai maksimal {WORKERS_HEAVY} worker agar RAM aman")
print("Konfigurasi siap. Output:", OUT.resolve(), "| checkpoint:", CKPT_DIR.resolve())

## 1. Pemuat dataset

- **Dataset simulasi (6)** dibaca dari file CSV hasil generate MOA oleh kelompok, di folder `Dataset Real Word/datasets_synthetic` (Sea, RBFBlips, LED_abrupt, LED_gradual, Tree, Hyperplane; masing-masing 100.000 baris, format `feature_0..N,target`, kelas di kolom terakhir). Drift sudah tertanam di dalam data sesuai pengaturan MOA saat generate.
- **Dataset real-world (4)** dibaca dari folder `dataset/` (electricity, kddcup99, covertype, weather).
- Catatan penamaan: folder `Dataset Real Word` itu sebenarnya berisi dataset **simulasi**; dataset real-world justru ada di folder `dataset`. Cukup diketahui saja, path di notebook mengikuti nama folder yang ada.

In [ ]:
def _load_moa(name, desc):
    df = pd.read_csv(SIM_DIR / f"{name}.csv")
    y = df["target"].astype(str)
    X = df.drop(columns=["target"])
    return X, y, 500, desc


def load_sea():
    return _load_moa("Sea", "SEA (MOA): gradual drift 25/50/75%, 100.000 baris, 3 fitur, 2 kelas")


def load_rbfblips():
    return _load_moa("RBFBlips", "RBFBlips (MOA): abrupt drift 25/50/75%, 100.000 baris, 20 fitur, 4 kelas")


def load_led_abrupt():
    return _load_moa("LED_abrupt", "LED_abrupt (MOA): abrupt drift di 50%, 100.000 baris, 24 fitur, 10 kelas")


def load_led_gradual():
    return _load_moa("LED_gradual", "LED_gradual (MOA): gradual drift 25/50/75%, 100.000 baris, 24 fitur, 10 kelas")


def load_tree():
    return _load_moa("Tree", "Tree (MOA): abrupt drift 25/50/75%, 100.000 baris, 30 fitur, 10 kelas")


def load_hyperplane():
    return _load_moa("Hyperplane", "Hyperplane (MOA): incremental drift t=0.001, 100.000 baris, 10 fitur, 2 kelas")


def load_electricity():
    df = pd.read_csv(DATA / "electricity" / "electricity.csv")
    y = df["class"]
    X = df.drop(columns=["date", "class"])
    return X, y, 500, "Elec2: 45.312 sampel, 7 fitur, 2 kelas (UP/DOWN)"


def load_covertype():
    df = pd.read_csv(DATA / "covertype" / "covtype.data", header=None)
    y = df.iloc[:, -1].astype(str)
    X = df.iloc[:, :-1]
    return X, y, 500, "Covertype: 581.012 sampel, 54 fitur, 7 kelas"


def load_kddcup99():
    df = pd.read_csv(DATA / "kddcup99" / "kdd_df.csv")
    y = df["target"]
    X = df.drop(columns=["target", "attack_type"])
    X = pd.get_dummies(X, columns=["protocol_type", "service", "flag"])
    # shortcut: kosong kategori dibangun dari seluruh file (ruang kategori, bukan label)
    return X, y, 500, "KDD Cup 1999 (10%): 494.020 sampel, 41 fitur (+one-hot), 23 kelas"


def load_weather():
    w = pd.read_csv(DATA / "weather" / "weather_prediction_dataset_light.csv")
    p = pd.read_csv(DATA / "weather" / "weather_prediction_picnic_labels.csv")
    cols = ["DATE"] + [c for c in w.columns if c.startswith("BASEL_") and not c.endswith("_wind_speed")]
    X = w[cols].merge(p[["DATE", "BASEL_picnic_weather"]], on="DATE")
    y = X["BASEL_picnic_weather"].map({True: "yes", False: "no"})
    X = X.drop(columns=["DATE", "BASEL_picnic_weather"])
    return X, y, 500, "Weather BASEL: 3.654 sampel, 9 fitur, 2 kelas (picnic yes/no); 8 chunk"


LOADERS = {
    "sea": load_sea,
    "rbfblips": load_rbfblips,
    "led_abrupt": load_led_abrupt,
    "led_gradual": load_led_gradual,
    "tree": load_tree,
    "hyperplane": load_hyperplane,
    "electricity": load_electricity,
    "weather": load_weather,
    "kddcup99": load_kddcup99,
    "covertype": load_covertype,
}
print("Loader siap:", ", ".join(LOADERS))

### Kapan drift terjadi di tiap dataset

Ingat: satu chunk = 500 baris, dan dataset simulasi berisi 100.000 baris = 200 chunk (chunk 1 = warm-up). Posisi drift 25% / 50% / 75% berarti perubahan konsep masuk pada **sampel ke-25.000 / 50.000 / 75.000**, yaitu **chunk ke-51, 101, dan 151**.

| Dataset | Jenis drift | Kapan berubah | Apa yang berubah |
|---|---|---|---|
| Sea | gradual | chunk 51, 101, 151 | fungsi klasifikasi berganti (ambang pada fitur berubah), transisi bertahap |
| RBFBlips | abrupt | chunk 51, 101, 151 | centroid klaster penghasil data diganti total, distribusi berubah mendadak |
| LED_abrupt | abrupt | chunk 101 saja | konfigurasi segmen LED yang rusak berubah, fitur relevan bergeser, sekali di tengah |
| LED_gradual | gradual | chunk 51, 101, 151 | sama seperti LED_abrupt tetapi perubahannya bertahap di tiga titik |
| Tree | abrupt | chunk 51, 101, 151 | struktur pohon acak penghasil label diganti pohon baru |
| Hyperplane | incremental | terus-menerus | batas hyperplane bergeser sedikit pada tiap sampel (t = 0.001), tidak ada titik tunggal |
| Electricity, Kddcup99, Covertype, Weather | real-world | tidak dikendalikan | drift alami bila ada; posisinya tidak diketahui sehingga dideteksi dari lonjakan error-rate |

Cara memakai tabel ini saat membaca hasil:
1. Di dataset simulasi, akurasi model seharusnya **anjlok di sekitar chunk 51 / 101 / 151** — itu efek drift.
2. Detektor CUSUM idealnya membunyikan alarm (tanda x merah) tak lama setelah titik tersebut. Alarm yang jauh sebelum/sesudahnya berarti terlalu sensitif / terlambat.
3. **Recovery time** dihitung dari chunk alarm sampai akurasi kembali ke minimal 95% baseline. Pada drift 75% (chunk 151), sisa stream hanya 50 chunk — pemulihan bisa gagal sebelum stream habis, dan itu memang bagian yang diuji.
4. Kalau akurasi tidak anjlok sama sekali di titik drift, model tersebut kebal terhadap drift itu (driftnya terlalu halus untuknya) — misalnya drift Sea yang hanya menggeser ambang.

## 2. Komponen: standardisasi online dan detektor drift

- `OnlineScaler`: standardisasi hanya dari data yang sudah dipakai latihan (mencegah *data leakage*).
- `WindowCUSUM`: skor θ gaya window CUSUM atas error-rate chunk relatif terhadap referensi; dipakai detektor luar (strategi) dan internal DACD.

In [ ]:
class OnlineScaler:
    """Standardisasi hanya dari data yang sudah dipakai latihan (hindari leakage)."""

    def __init__(self):
        self.n = 0
        self.mean = None
        self.m2 = None

    def transform(self, X):
        arr = np.asarray(X, dtype=float)
        if self.mean is None:
            return arr
        return (arr - self.mean) / np.sqrt(self.var + 1e-9)

    def update(self, X):
        arr = np.asarray(X, dtype=float)
        if self.mean is None:
            self.mean = np.zeros(arr.shape[1])
            self.m2 = np.zeros(arr.shape[1])
        for row in arr:
            self.n += 1
            delta = row - self.mean
            self.mean += delta / self.n
            self.m2 += delta * (row - self.mean)

    @property
    def var(self):
        return self.m2 / max(self.n, 1)


class WindowCUSUM:
    """Skor gaya window CUSUM atas error-rate chunk relatif terhadap referensi."""

    def __init__(self, k=0.10, lambd=0.99, threshold=THETA_SEVERE):
        self.k = k
        self.lambd = lambd
        self.threshold = threshold
        self.s = 0.0
        self.ref_error = None
        self.ref_n = 0

    def update_ref(self, error):
        if self.ref_error is None:
            self.ref_error = error
            self.ref_n = 1
        else:
            self.ref_error = (self.ref_error * self.ref_n + error) / (self.ref_n + 1)
            self.ref_n = min(self.ref_n + 1, 30)

    def score(self, error):
        if self.ref_error is None or self.ref_error <= 0:
            self.update_ref(error)
            return 0.0
        excess = (error - self.ref_error) / self.ref_error - self.k
        self.s = max(0.0, self.lambd * self.s + excess)
        return self.s

    def alarm(self, theta):
        return theta >= self.threshold


print("OnlineScaler dan WindowCUSUM siap.")

## 3. Tujuh model

Antarmuka seragam: `test_chunk(X, y)`, `train_chunk(X, y)`, `reset(seed)`.

- **DT / HAT / ARF**: model river, test-then-train per sampel di dalam chunk.
- **SVM / ANN**: scikit-learn dengan `partial_fit`/`fit` per chunk.
- **BELS (ringkas)**: pemetaan acak bersama (tanh) + ensemble lapisan output ridge; anggota terlemah dibuang (passive ensemble, mini-chunk) mengikuti ide paper BELS.
- **DACD (ringkas)**: MLP + CUSUM internal; saat alarm, model dilatih ulang atas buffer + chunk dengan oversampling kelas minoritas, mengikuti ide paper DACD.

In [ ]:
class RiverModel:
    """HT/HAT/ARF: test-then-train per sampel di dalam chunk. spec disimpan agar bisa di-pickle."""

    def __init__(self, name, spec, seed):
        self.name = name
        self.spec = spec
        self.seed = seed
        self.model = self._build()

    def _build(self):
        if self.spec == "ht":
            return tree.HoeffdingTreeClassifier(grace_period=100, delta=1e-4)
        if self.spec == "hat":
            return tree.HoeffdingAdaptiveTreeClassifier(seed=self.seed, grace_period=100, delta=1e-4)
        return forest.ARFClassifier(n_models=ARF_TREES, seed=self.seed)

    def test_chunk(self, X, y):
        hits = 0
        preds = []
        for xi, yi in zip(X.to_dict("records"), y):
            p = self.model.predict_one(xi)
            preds.append(p)
            hits += p == yi
        return np.array(preds), hits / len(y)

    def train_chunk(self, X, y):
        for xi, yi in zip(X.to_dict("records"), y):
            self.model.learn_one(xi, yi)

    def reset(self, seed):
        self.model = self._build()


class SklearnModel:
    def __init__(self, seed, kind):
        self.seed = seed
        self.kind = kind
        self.scaler = OnlineScaler()
        self.classes = None
        self.buffer = {}
        self.fitted_classes = None
        self.model = self._new()

    def _new(self):
        if self.kind == "svm":
            return SGDClassifier(loss="hinge", alpha=1e-4, random_state=self.seed)
        return MLPClassifier(hidden_layer_sizes=(32,), random_state=self.seed, max_iter=1, warm_start=True)

    def test_chunk(self, X, y):
        Xs = self.scaler.transform(X)
        preds = self.model.predict(Xs)
        return preds, float(np.mean(np.asarray(y).astype(str) == preds.astype(str)))

    def _fit_incremental(self, X, y_arr):
        known = self.fitted_classes or sorted(set(y_arr))
        missing = [c for c in known if c not in set(y_arr)]
        if missing:
            pad = pd.DataFrame([self.buffer[c] for c in missing if c in self.buffer], columns=X.columns)
            keep = [c for c in missing if c in self.buffer]
            if len(pad):
                X = pd.concat([X, pad], ignore_index=True)
                y_arr = np.concatenate([y_arr, np.array(keep)])
        Xs = self.scaler.transform(X)
        try:
            self.model.fit(Xs, y_arr)
        except ValueError:
            self.model = self._new()
            self.model.fit(Xs, y_arr)
        self.fitted_classes = sorted(set(y_arr))

    def train_chunk(self, X, y):
        y_arr = np.asarray(y).astype(str)
        if self.kind == "svm":
            Xs = self.scaler.transform(X)
            self.model.partial_fit(Xs, y_arr, classes=self.classes)
        else:
            for c in np.unique(y_arr):
                self.buffer[c] = X.loc[y_arr == c].iloc[-1]
            self._fit_incremental(X, y_arr)
        self.scaler.update(X)

    def reset(self, seed):
        self.scaler = OnlineScaler()
        self.buffer = {}
        self.fitted_classes = None
        self.model = self._new()


class BELSModel:
    """BELS ringkas: ensemble lapisan output ridge di atas pemetaan acak bersama."""

    name = "BELS"

    def __init__(self, seed, n_z=60, n_h=40, ridge=1.0, max_members=8, keep=4):
        self.seed = seed
        self.n_z, self.n_h, self.ridge = n_z, n_h, ridge
        self.max_members, self.keep = max_members, keep
        self.scaler = OnlineScaler()
        self.classes = None
        self.rng = np.random.default_rng(seed)
        self.Wz = self.bz = self.Wh = self.bh = None
        self.members = []

    def _init_map(self, d):
        self.Wz = self.rng.normal(0, 0.5, (d, self.n_z))
        self.bz = self.rng.normal(0, 0.1, self.n_z)
        self.Wh = self.rng.normal(0, 0.5, (self.n_z, self.n_h))
        self.bh = self.rng.normal(0, 0.1, self.n_h)

    def _map(self, Xs):
        Z = np.tanh(Xs @ self.Wz + self.bz)
        H = np.tanh(Z @ self.Wh + self.bh)
        return np.hstack([Z, H])

    def test_chunk(self, X, y):
        if not self.members:
            preds = np.array([str(self.classes[0])] * len(y), dtype=object)
            return preds, float(np.mean(np.asarray(y).astype(str) == preds))
        A = self._map(self.scaler.transform(X))
        scores = np.mean([A @ beta for beta, _ in self.members], axis=0)
        preds = np.asarray(self.classes, dtype=object)[scores.argmax(axis=1)]
        return preds, float(np.mean(np.asarray(y).astype(str) == preds))

    def train_chunk(self, X, y):
        y_arr = np.asarray(y).astype(str)
        Xs = self.scaler.transform(X)
        if self.Wz is None:
            self._init_map(Xs.shape[1])
        A = self._map(Xs)
        Y = (y_arr[:, None] == np.asarray(self.classes)[None, :]).astype(float)
        target = Y.argmax(axis=1)
        accs = []
        for beta, ema in self.members:
            acc = float(np.mean((A @ beta).argmax(axis=1) == target))
            accs.append(0.7 * ema + 0.3 * acc)
        G = A.T @ A + self.ridge * np.eye(A.shape[1])
        beta_new = np.linalg.solve(G, A.T @ Y)
        acc_new = float(np.mean((A @ beta_new).argmax(axis=1) == target))
        self.members = [(b, a) for (b, _), a in zip(self.members, accs)] + [(beta_new, acc_new)]
        if len(self.members) > self.max_members:
            self.members = sorted(self.members, key=lambda m: m[1], reverse=True)[: self.keep]
        self.scaler.update(X)

    def reset(self, seed):
        self.rng = np.random.default_rng(seed)
        self.Wz = self.bz = self.Wh = self.bh = None
        self.members = []
        self.scaler = OnlineScaler()


class DACDModel:
    """DACD ringkas: window CUSUM internal + pemilihan sampel (buffer, oversampling) + retrain MLP."""

    name = "DACD"

    def __init__(self, seed, hidden=32, buffer_size=2000):
        self.seed = seed
        self.hidden = hidden
        self.buffer_size = buffer_size
        self.scaler = OnlineScaler()
        self.classes = None
        self.fitted_classes = None
        self.buffer = []
        self.cusum = WindowCUSUM()
        self.model = MLPClassifier(hidden_layer_sizes=(hidden,), random_state=seed, max_iter=1, warm_start=True)

    def test_chunk(self, X, y):
        Xs = self.scaler.transform(X)
        preds = self.model.predict(Xs)
        return preds, float(np.mean(np.asarray(y).astype(str) == preds.astype(str)))

    def _fit(self, Xs, y_arr, fresh=False, max_iter=1):
        if fresh:
            self.model = MLPClassifier(hidden_layer_sizes=(self.hidden,), random_state=self.seed, max_iter=max_iter)
        try:
            self.model.fit(Xs, y_arr)
        except ValueError:
            self.model = MLPClassifier(hidden_layer_sizes=(self.hidden,), random_state=self.seed, max_iter=max_iter)
            self.model.fit(Xs, y_arr)
        self.fitted_classes = sorted(set(y_arr))

    def train_chunk(self, X, y):
        y_arr = np.asarray(y).astype(str)
        for i in range(len(y_arr)):
            self.buffer.append((X.iloc[i].to_numpy(dtype=float), y_arr[i]))
        if len(self.buffer) > self.buffer_size:
            del self.buffer[: len(self.buffer) - self.buffer_size]
        Xs = self.scaler.transform(X)
        fitted = True
        try:
            err = 1.0 - float(np.mean(self.model.predict(Xs) == y_arr))
        except ValueError:
            fitted = False
            err = None
        theta = self.cusum.score(err) if fitted else 0.0
        if fitted and self.cusum.alarm(theta):
            buf_X = np.array([b[0] for b in self.buffer])
            buf_y = np.array([b[1] for b in self.buffer])
            counts = {c: int(np.sum(buf_y == c)) for c in np.unique(buf_y)}
            med = int(np.median(list(counts.values())))
            extra = []
            for c, cnt in counts.items():
                if cnt < med:
                    idxs = np.where(buf_y == c)[0]
                    take = self.seed + len(extra)
                    extra.extend(np.random.default_rng(take).choice(idxs, med - cnt, replace=True).tolist())
            if extra:
                buf_X = np.vstack([buf_X, buf_X[extra]])
                buf_y = np.concatenate([buf_y, buf_y[extra]])
            self._fit(self.scaler.transform(buf_X), buf_y, fresh=True, max_iter=3)
            self.cusum = WindowCUSUM()
        else:
            if fitted:
                self.cusum.update_ref(err)
            known = self.fitted_classes or sorted(set(y_arr))
            missing = [c for c in known if c not in set(y_arr)]
            if missing:
                pad = [b[0] for b in self.buffer[::-1] if b[1] in missing][: len(missing)]
                pad_y = [b[1] for b in self.buffer[::-1] if b[1] in missing][: len(missing)]
                if pad:
                    Xs = np.vstack([Xs, self.scaler.transform(np.array(pad))])
                    y_arr = np.concatenate([y_arr, np.array(pad_y)])
            self._fit(Xs, y_arr)
        self.scaler.update(X)

    def reset(self, seed):
        self.scaler = OnlineScaler()
        self.buffer = []
        self.cusum = WindowCUSUM()
        self.fitted_classes = None
        self.model = MLPClassifier(hidden_layer_sizes=(self.hidden,), random_state=seed, max_iter=1, warm_start=True)


def make_models(seed):
    svm = SklearnModel(seed, "svm")
    svm.name = "SVM"
    ann = SklearnModel(seed, "ann")
    ann.name = "ANN"
    return [
        RiverModel("DT", "ht", seed),
        RiverModel("HAT", "hat", seed),
        RiverModel("ARF", "arf", seed),
        svm,
        ann,
        BELSModel(seed),
        DACDModel(seed),
    ]


print("7 model siap: DT, HAT, ARF, SVM, ANN, BELS, DACD.")

## 4. Fungsi pengujian (paralel + resume)

Satu run = satu model × satu strategi. Ke-14 run per dataset dijalankan **paralel di proses terpisah** memakai joblib.

**Penyesuaian device otomatis**: jumlah worker = jumlah thread CPU dikurangi `RESERVE_THREADS` (default 2, disisakan untuk sistem/navigasi). Contoh: Ryzen 9 5980HS 8 core/16 thread → 14 worker. Dataset berat (kddcup99, covertype) dibatasi `WORKERS_HEAVY` agar RAM tidak jebol. Setiap worker membatasi BLAS 1 thread (threadpoolctl) supaya tidak oversubscription. Ubah `RESERVE_THREADS`/`N_WORKERS` di sel konfigurasi bila perlu.

**Mekanisme resume** (kalau mati listrik / kernel mati / Run All dihentikan):
- Setiap selesai satu chunk, state run disimpan sebagai checkpoint atomik di `hasil_pengujian/checkpoint/` (model, kappa, CUSUM, riwayat chunk).
- Saat dijalankan ulang, run yang sudah selesai dilewati (`[selesai sebelumnya]`), run yang setengah jalan dilanjutkan dari chunk terakhir (`[lanjut]`), dan checkpoint yang rusak diulang dari awal run itu.
- Resume valid karena semua dataset deterministik: seluruh dataset (simulasi MOA maupun real-world) dibaca dari file yang sama.
- Log per run juga tersimpan di `hasil_pengujian/logs/<dataset>_<model>_<strategi>.log` agar progres tetap bisa dipantau saat paralel.
- Bila muncul `[peringatan] checkpoint ... gagal ditulis`, berarti file checkpoint sedang dikunci proses lain (kernel lama yang masih jalan, antivirus, atau OneDrive). Pastikan hanya satu kernel menjalankan notebook ini: **Interrupt lalu Restart kernel** sebelum Run All ulang.
- Untuk memulai semua dari nol: hapus folder `hasil_pengujian`.

In [ ]:
def _atomic_replace(src, dst, tries=6):
    """os.replace tahan kunci file (kernel lain/antivirus/OneDrive): retry, lalu fallback, terakhir menyerah tanpa crash."""
    for i in range(tries):
        try:
            os.replace(src, dst)
            return True
        except PermissionError:
            time.sleep(0.25 * (i + 1))
    try:
        if dst.exists():
            dst.unlink()
    except PermissionError:
        pass
    try:
        os.replace(src, dst)
        return True
    except PermissionError:
        print(f"    [peringatan] checkpoint chunk ini gagal ditulis (file terkunci): {dst.name}")
        try:
            src.unlink()
        except OSError:
            pass
        return False


def _log(tag, msg):
    """Cetak ke layar dan simpan ke file log per run (agar progres tetap terlihat saat paralel)."""
    print(msg, flush=True)
    try:
        with open(LOG_DIR / f"{tag}.log", "a", encoding="utf-8") as f:
            f.write(msg + "\n")
    except OSError:
        pass


def run_one(model, X, y, chunk, strategy, seed, classes, tag, log_every=10):
    csv_path = OUT / f"{tag}_chunks.csv"
    sum_path = OUT / f"{tag}_summary.json"
    ckpt_path = CKPT_DIR / f"{tag}.pkl"

    if csv_path.exists() and sum_path.exists() and not ckpt_path.exists():
        df = pd.read_csv(csv_path)
        with open(sum_path, encoding="utf-8") as f:
            summary = json.load(f)
        _log(tag, f"    [selesai sebelumnya] {tag}: avg_acc={summary['avg_acc']:.4f}")
        return df, summary

    kappa = metrics.CohenKappa()
    cusum = WindowCUSUM()
    rows = []
    alarms = []
    chunk_id = 0
    if ckpt_path.exists():
        try:
            with open(ckpt_path, "rb") as f:
                state = cloudpickle.load(f)
            model, kappa, cusum = state["model"], state["kappa"], state["cusum"]
            rows, alarms, chunk_id = state["rows"], state["alarms"], state["chunk_id"]
            _log(tag, f"    [lanjut] {tag} dari chunk {chunk_id + 1}")
        except Exception:
            rows, alarms, chunk_id = [], [], 0
            _log(tag, f"    [checkpoint rusak, run diulang dari awal] {tag}")

    if hasattr(model, "classes"):
        model.classes = classes
    n = len(y)
    total_chunks = (n + chunk - 1) // chunk
    start = chunk_id * chunk
    while start < n:
        end = min(start + chunk, n)
        Xc, yc = X.iloc[start:end], y.iloc[start:end]
        chunk_id += 1
        t0 = time.perf_counter()
        if chunk_id == 1:
            model.train_chunk(Xc, yc)
            rows.append({"chunk": chunk_id, "n": end - start, "acc": np.nan, "kappa": np.nan,
                         "theta": 0.0, "alarm": False, "latency_ms": (time.perf_counter() - t0) * 1000})
        else:
            preds, acc = model.test_chunk(Xc, yc)
            for yi, pi in zip(yc, preds):
                kappa.update(str(yi), str(pi))
            error = 1.0 - acc
            theta = cusum.score(error)
            alarm = cusum.alarm(theta)
            if alarm:
                alarms.append(chunk_id)
            if strategy == "reset" and alarm:
                model.reset(seed)
                cusum = WindowCUSUM()
            else:
                cusum.update_ref(error)
            model.train_chunk(Xc, yc)
            rows.append({"chunk": chunk_id, "n": end - start, "acc": acc, "kappa": kappa.get(),
                         "theta": theta, "alarm": alarm, "latency_ms": (time.perf_counter() - t0) * 1000})
        if chunk_id % log_every == 0 or chunk_id == total_chunks:
            r = rows[-1]
            acc_txt = "warm-up" if np.isnan(r["acc"]) else f"{r['acc']:.3f}"
            _log(tag, f"    [{model.name} / {strategy}] chunk {chunk_id:>4}/{total_chunks} acc={acc_txt} theta={r['theta']:.2f}")
        tmp_path = ckpt_path.with_suffix(".tmp")
        with open(tmp_path, "wb") as f:
            cloudpickle.dump({"model": model, "kappa": kappa, "cusum": cusum, "rows": rows,
                              "alarms": alarms, "chunk_id": chunk_id}, f)
        _atomic_replace(tmp_path, ckpt_path)
        start = end
    df = pd.DataFrame(rows)

    accs = df["acc"].dropna()
    baseline_idx = df[(df["chunk"] > 1) & (~df["alarm"])].head(max(3, int(len(df) * 0.1) + 1))
    baseline = float(baseline_idx["acc"].mean()) if len(baseline_idx) else float(accs.iloc[0])
    threshold = RECOVERY_RATIO * baseline
    eval_df = df.dropna(subset=["acc"]).copy()
    eval_df["degraded"] = eval_df["acc"] < threshold

    episodes = int((eval_df["degraded"] & ~eval_df["degraded"].shift(1, fill_value=False)).sum())
    ftde = float(eval_df["degraded"].mean())
    pde = episodes / max(len(eval_df), 1)
    rt = None
    for a in alarms:
        after = eval_df[eval_df["chunk"] > a]
        ok = after[~after["degraded"]]
        if len(ok):
            rt = int(ok["chunk"].iloc[0] - a)
            break
    summary = {
        "model": model.name, "strategy": strategy, "chunks": len(df),
        "avg_acc": float(accs.mean()),
        "final_kappa": float(df["kappa"].dropna().iloc[-1]) if len(df["kappa"].dropna()) else np.nan,
        "baseline": baseline, "threshold": threshold, "alarms": len(alarms),
        "pde": pde, "ftde": ftde, "rt_chunks": rt if rt is not None else -1,
        "avg_latency_ms": float(df["latency_ms"].mean()),
    }
    df.to_csv(csv_path, index=False)
    with open(sum_path, "w", encoding="utf-8") as f:
        json.dump(summary, f)
    if ckpt_path.exists():
        ckpt_path.unlink()
    return df, summary


def plot_curves(name, curves):
    models = sorted({label.split(" / ")[0] for label in curves})
    ncols = 2
    nrows = (len(models) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3.2 * nrows), sharex=True, sharey=True, squeeze=False)
    cmap = matplotlib.colormaps["tab10"]
    for i, model in enumerate(models):
        ax = axes[i // ncols][i % ncols]
        color = cmap(i % 10)
        for strategy, ls in (("continuous", "-"), ("reset", "--")):
            df = curves.get(f"{model} / {strategy}")
            if df is None:
                continue
            d = df.dropna(subset=["acc"])
            ax.plot(d["chunk"], d["acc"], marker="o", markersize=2.5, linewidth=1.3,
                    linestyle=ls, color=color, label=strategy)
            alarms = d[d["alarm"]]
            if len(alarms):
                ax.scatter(alarms["chunk"], alarms["acc"], marker="x", s=24, color="red", linewidths=1.1, zorder=5)
        ax.set_title(model, fontsize=10)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7, loc="lower right")
    for j in range(len(models), nrows * ncols):
        axes[j // ncols][j % ncols].set_visible(False)
    fig.suptitle(f"Prequential: {name} — garis penuh = continuous, putus-putus = reset, x merah = alarm drift", fontsize=11)
    fig.supxlabel("Chunk")
    fig.supylabel("Akurasi real-time")
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(OUT / f"plot_{name}.png", dpi=150)
    plt.show()
    return fig


ALL_SUMMARIES = []
MODEL_NAMES = None


def _run_task(name, model_index, strategy):
    """Satu tugas paralel = satu run (model x strategi) pada satu dataset; jalan di proses terpisah."""
    X, y, chunk, desc = LOADERS[name]()
    classes = np.array(sorted(y.astype(str).unique()))
    model = make_models(SEED)[model_index]
    tag = f"{name}_{model.name}_{strategy}".replace(" ", "")
    _log(tag, f"=== {name} | {desc} | chunk={chunk} | baris={len(y)} | mulai {model.name}/{strategy}")
    try:
        from threadpoolctl import threadpool_limits
        limits = threadpool_limits(1)
    except ImportError:
        from contextlib import nullcontext
        limits = nullcontext()
    with limits:
        df, summary = run_one(model, X, y, chunk, strategy, SEED, classes, tag)
    return summary


def run_dataset(name):
    global MODEL_NAMES
    if MODEL_NAMES is None:
        MODEL_NAMES = [m.name for m in make_models(SEED)]
    tasks = [(i, s) for i in range(len(MODEL_NAMES)) for s in STRATEGIES]
    if name in ("kddcup99", "covertype"):
        n_jobs = min(WORKERS_HEAVY, len(tasks))
    else:
        n_jobs = min(N_WORKERS, len(tasks))
    print(f"=== {name} | {len(tasks)} run paralel | {n_jobs} worker "
          f"(CPU {TOTAL_THREADS} thread, {RESERVE_THREADS} disisakan) ===", flush=True)
    results = Parallel(n_jobs=n_jobs, verbose=1)(delayed(_run_task)(name, i, s) for i, s in tasks)
    curves = {}
    rows = []
    for summary in results:
        tag = f"{name}_{summary['model']}_{summary['strategy']}".replace(" ", "")
        curves[f"{summary['model']} / {summary['strategy']}"] = pd.read_csv(OUT / f"{tag}_chunks.csv")
        ALL_SUMMARIES.append({"dataset": name, **summary})
        rows.append(summary)
        print(f"  >> {summary['model']:<5} {summary['strategy']:<10} avg_acc={summary['avg_acc']:.4f} "
              f"kappa={summary['final_kappa']:.4f} alarm={summary['alarms']} RT={summary['rt_chunks']} "
              f"PDE={summary['pde']:.3f} FTDE={summary['ftde']:.3f} lat={summary['avg_latency_ms']:.1f}ms", flush=True)
    plot_curves(name, curves)
    return pd.DataFrame(rows).set_index(["model", "strategy"])


print("Fungsi pengujian siap (paralel + resume).")

## SEA — gradual drift 25/50/75% (drift di chunk 51, 101, 151)

In [ ]:
run_dataset("sea")

## RBFBlips — abrupt drift 25/50/75% (drift di chunk 51, 101, 151)

In [ ]:
run_dataset("rbfblips")

## LED abrupt — drift tepat di 50% (chunk 101)

In [ ]:
run_dataset("led_abrupt")

## LED gradual — drift 25/50/75% (chunk 51, 101, 151)

In [ ]:
run_dataset("led_gradual")

## Tree — abrupt drift 25/50/75% (chunk 51, 101, 151)

In [ ]:
run_dataset("tree")

## Hyperplane — incremental drift (bergesar tiap sampel, tanpa titik tunggal)

In [ ]:
run_dataset("hyperplane")

## Electricity (Elec2) — real-world, 45.312 baris; posisi drift tidak diketahui (drift alami)

In [ ]:
run_dataset("electricity")

## Weather (BASEL) — real-world, 3.654 baris, 8 chunk; drift alami bila ada

In [ ]:
run_dataset("weather")

## KDD Cup 1999 (10%) — real-world, 494.020 baris = 988 chunk; drift alami, ±20-30 menit

In [ ]:
run_dataset("kddcup99")

## Covertype — real-world, 581.012 baris = 1.163 chunk; drift alami, sel paling lama

In [ ]:
run_dataset("covertype")

## Ringkasan semua run

In [ ]:
ringkasan = pd.DataFrame(ALL_SUMMARIES)
ringkasan.to_csv(OUT / "ringkasan_semua.csv", index=False)
ringkasan.set_index(["dataset", "model", "strategy"])[
    ["avg_acc", "final_kappa", "alarms", "rt_chunks", "pde", "ftde", "avg_latency_ms"]
].round(4)

## Catatan

- Dataset simulasi adalah file CSV hasil generate MOA asli oleh kelompok (folder `Dataset Real Word/datasets_synthetic`), sehingga mekanisme drift (gradual/abrupt/incremental dan posisinya) persis mengikuti pengaturan MOA saat generate, bukan rekonstruksi.
- Nama folder `Dataset Real Word` agak menyesatkan: isinya dataset **simulasi** MOA; dataset **real-world** ada di folder `dataset/`. Pertimbangkan rename folder bila ingin lebih rapi (lalu sesuaikan `SIM_DIR` di sel konfigurasi).
- BELS dan DACD di sini implementasi ringkas dari ide inti paper (BELS: ensemble lapisan output mini-chunk; DACD: CUSUM + pemilihan sampel + retrain NN), bukan kode resmi penulis.
- `rt_chunks = -1` berarti tidak ada alarm drift sehingga recovery time tidak terdefinisi.
- Dataset Weather di slide tertulis 95.100 sampel dan 3 kelas; dataset Zenodo aslinya 3.654 baris dan label biner. Angka slide perlu dikoreksi.
- Covertype dan KDD adalah data statis yang dibaca berurutan sebagai stream (praktik umum, dengan limitasi urutan waktu tidak asli).